# 06 Forecast Report Workflow

A forecasting report should do more than print model output. It should explain the data pattern, justify the model family, show the fitted model behavior, report accuracy measures, and present forecasts in a table that a decision maker can use.

In [ ]:
from lite_setup import ensure_packages
await ensure_packages()

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from checks import check_between, check_close, check_columns
from smoothing_utils import (
    accuracy_measures, initial_level_mean, initial_line,
    simple_es, optimize_simple_es,
    holt_trend, optimize_holt, holt_forecast_table,
    holt_winters, optimize_holt_winters, holt_winters_forecast,
)

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.precision', 4)
DATA_DIR = Path('data')

## Transfer Checklist

1. Plot the time series in time order.
2. Identify level, trend, and seasonality.
3. Decide whether seasonal variation looks additive or multiplicative.
4. Choose a smoothing model family.
5. State how initial values are chosen.
6. Fit or optimize smoothing constants.
7. Report SSE, MAD, MSE, and MAPE.
8. Plot observed values and one-step fitted values.
9. Forecast the requested horizon and report intervals when the course formula supports them.
10. Write a short interpretation that connects the numbers to the problem.

In [ ]:
calc = pd.read_csv(DATA_DIR / 'calculator_sales_hw8.csv')
y = calc['Sales'].to_numpy()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(calc['Time'], y, marker='o')
ax.set_xlabel('Month')
ax.set_ylabel('Calculator sales')
ax.set_title('Calculator sales for two years')
plt.tight_layout()

The calculator sales series has an upward trend and no obvious repeated monthly seasonal pattern from only two years of data. This makes Holt's trend-corrected model a reasonable course-model choice.

In [ ]:
l0, b0 = initial_line(y, n=12)
alpha, gamma, fit, metrics = optimize_holt(y, l0=l0, b0=b0)
forecast = holt_forecast_table(fit, alpha=alpha, gamma=gamma, h=3)

print(f'Initial l0={l0:.3f}, b0={b0:.3f}')
print(f'Optimized alpha={alpha:.4f}, gamma={gamma:.4f}')
print('Accuracy measures')
display(pd.Series(metrics).to_frame('value'))
print('Forecast table')
display(forecast.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(calc['Time'], y, marker='o', label='Observed')
ax.plot(calc['Time'], fit['yhat_one_step'], marker='o', label='One-step Holt forecast')
future_time = np.arange(len(y) + 1, len(y) + 4)
ax.plot(future_time, forecast['forecast'], marker='o', label='Forecast')
ax.fill_between(future_time, forecast['lower_95_PI'], forecast['upper_95_PI'], alpha=0.2, label='Approx. 95% PI')
ax.set_xlabel('Month')
ax.set_ylabel('Calculator sales')
ax.legend()
plt.tight_layout()

## Report Skeleton

A concise report could say: the series shows an increasing trend, so Holt's trend-corrected exponential smoothing is appropriate. The initial level and trend were estimated by fitting a line to the first year. The smoothing constants were selected to minimize SSE subject to `0 <= alpha <= 1` and `0 <= gamma <= 1`. Forecasts are increasing because the final trend estimate is positive, and prediction intervals quantify uncertainty for individual future monthly sales.